# Chapter 12 · Tokenizers and the hidden tax — Lab 12

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/lab12_tokenizers.ipynb)

**Measure the tax, then set it yourself.** With English, Spanish and Portuguese slices from FineWeb-Edu and FineWeb2 you will: measure fertility and bytes-per-token for four open tokenizers and compute the tax ratio per language; re-state Lab 10's data mix in tokens; train the chapter's from-scratch BPE on two language mixes and print the real merge tables; train a production BPE with the `tokenizers` library and compare fertility; compute the embedding and softmax bill as a function of vocabulary size; audit the vocabulary's tail; and extend a base vocabulary with initialized rows and measure the fertility drop. CPU is enough; ~15 minutes.

*Omar U. Florez, PhD · Efficient Training of LLMs · Lab 12*

In [ ]:
%pip -q install datasets transformers tokenizers torch pyarrow
import re, json, time, collections, itertools, pathlib, random
import numpy as np, matplotlib.pyplot as plt
from datasets import load_dataset

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
N_PER_LANG = 1500
random.seed(0)

## 0 · Three languages, one warehouse

English from FineWeb-Edu (Lab 10's slice if present), Spanish and Portuguese from FineWeb2 — the per-language pipeline of Chapter 10.6. We keep ~1,500 documents per language and hold out 200 of each for the fertility measurements.

In [ ]:
def take(stream, n): return [d["text"] for d in itertools.islice(stream, n)]
p = pathlib.Path("fineweb_edu_lab10/data.parquet")
if p.exists():
    import pyarrow.parquet as pq
    en = [r["text"] for r in pq.read_table(p, columns=["text"]).to_pylist()[:N_PER_LANG]]
else:
    en = take(load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True), N_PER_LANG)
es = take(load_dataset("HuggingFaceFW/fineweb-2", name="spa_Latn", split="train", streaming=True), N_PER_LANG)
pt = take(load_dataset("HuggingFaceFW/fineweb-2", name="por_Latn", split="train", streaming=True), N_PER_LANG)
corpus = {"en": en, "es": es, "pt": pt}
train = {k: v[200:] for k, v in corpus.items()}; test = {k: v[:200] for k, v in corpus.items()}
for k, v in corpus.items(): print(k, len(v), "docs;", sum(len(d.split()) for d in v), "words")
print("\n", es[0][:300].replace("\n", " "))

## 1 · The tax, measured: fertility and bytes per token

Chapter 12.1's two numbers for four open tokenizers of different generations and mixes. (Llama 3 and Gemma 3 are gated; add them with a Hugging Face token if you have access — the code is the same.) The last column is the tax: fertility relative to English under the same tokenizer.

In [ ]:
from transformers import AutoTokenizer
TOKS = {"SmolLM2 · 49K (en-centric)": "HuggingFaceTB/SmolLM2-1.7B",
        "DeepSeek-V3 · 128K": "deepseek-ai/DeepSeek-V3",
        "Qwen2.5 · 151K": "Qwen/Qwen2.5-7B",
        "BLOOM · 250K (multilingual, es-rich)": "bigscience/bloom"}
# TOKS["Llama 3 · 128K"] = "meta-llama/Llama-3.1-8B"   # gated: huggingface-cli login first
def measure(tok, docs):
    n_tok = sum(len(tok(d, add_special_tokens=False).input_ids) for d in docs)
    return n_tok / sum(len(d.split()) for d in docs), sum(len(d.encode()) for d in docs) / n_tok
rows = []
for name, repo in TOKS.items():
    try: tok = AutoTokenizer.from_pretrained(repo)
    except Exception as e: print("skip", name, e); continue
    m = {lang: measure(tok, test[lang]) for lang in test}
    for lang in test: rows.append((name, len(tok), lang, m[lang][0], m[lang][1], m[lang][0] / m["en"][0]))
import pandas as pd
df = pd.DataFrame(rows, columns=["tokenizer", "V", "lang", "fertility", "bytes/token", "ratio vs en"])
print(df.round(3).to_string(index=False))
piv = df.pivot(index="tokenizer", columns="lang", values="ratio vs en")[["en", "es", "pt"]]
piv.plot.bar(figsize=(8, 3.2), color=[GRAY, AMBER, BLUE]); plt.axhline(1, color=GRAY, lw=1)
plt.ylabel("fertility ÷ English"); plt.title("Figure 12.1, measured on your slices"); plt.xticks(rotation=0); plt.legend(title="");

> **Read the bars.** The tax falls as the vocabulary grows and as its training mix includes the language — BLOOM, trained with a large Spanish share, should be the flattest. Note the bytes/token column: a tokenizer can have *lower* fertility on Spanish than on English simply because Spanish words are longer; the ratio is what the chapter calls the tax.

## 2 · The data mix, re-stated in tokens

Chapter 12.1's worked example on your numbers: the document share of each language versus its token share under each tokenizer.

In [ ]:
doc_share = {"en": 0.90, "es": 0.07, "pt": 0.03}          # a plausible Chapter-10 mix by documents
print("document share:", doc_share)
for name in piv.index:
    f = df[df.tokenizer == name].set_index("lang")["fertility"]
    w = {l: doc_share[l] * f[l] for l in doc_share}; tot = sum(w.values())
    print(f"{name:>38}: token share " + "  ".join(f"{l} {w[l]/tot:.1%}" for l in w))

## 3 · BPE from scratch, on two mixes

The chapter's algorithm, verbatim: training returns an ordered merge list; encoding replays it. We train two 8K vocabularies — English-centric (95/3/2) and Latin-weighted (60/25/15) — on the same number of documents, print the first merges of each and the Spanish-specific merges that only the second one learns, then measure fertility per language under both.

In [ ]:
PAT = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\w\s]+|\s+")      # GPT-2-style pieces
def pretokenize(text): return PAT.findall(text)
def merge_pair(sym, a, b):
    out, i = [], 0
    while i < len(sym):
        if i < len(sym) - 1 and sym[i] == a and sym[i + 1] == b: out.append(a + b); i += 2
        else: out.append(sym[i]); i += 1
    return tuple(out)
def train_bpe(texts, vocab_size, log_every=0):
    pieces = collections.Counter()
    for t in texts:
        for w in pretokenize(t): pieces[tuple(bytes([c]) for c in w.encode())] += 1
    n_vocab, merges, counts = 256, [], []
    while n_vocab < vocab_size:
        pairs = collections.Counter()
        for sym, n in pieces.items():
            for a, b in zip(sym, sym[1:]): pairs[(a, b)] += n
        if not pairs: break
        a, b = max(pairs, key=pairs.get); merges.append((a, b)); counts.append(pairs[(a, b)]); n_vocab += 1
        pieces = collections.Counter({merge_pair(s, a, b): n for s, n in pieces.items()})
        if log_every and len(merges) % log_every == 0: print(f"  {len(merges)} merges, last = {(a+b)!r} ×{pairs[(a,b)]}")
    return merges, counts
def encode(text, merges):
    toks = []
    for w in pretokenize(text):
        sym = tuple(bytes([c]) for c in w.encode())
        for a, b in merges: sym = merge_pair(sym, a, b)
        toks.extend(sym)
    return toks
def mix(weights, n=600):
    out = []
    for lang, w in weights.items(): out += random.sample(train[lang], min(len(train[lang]), int(n * w)))
    random.shuffle(out); return out
V = 256 + 2000                               # 2,000 merges: enough to see the effect on CPU in ~2 min
t0 = time.time()
m_en, c_en = train_bpe(mix({"en": .95, "es": .03, "pt": .02}), V)
m_lat, c_lat = train_bpe(mix({"en": .60, "es": .25, "pt": .15}), V)
print(f"two vocabularies in {time.time()-t0:.0f}s")
def show(merges, counts, k=12):
    for i, ((a, b), n) in enumerate(zip(merges[:k], counts[:k]), 1): print(f"{i:>3}  {a.decode(errors='replace')!r} + {b.decode(errors='replace')!r} → {(a+b).decode(errors='replace')!r}   ×{n}")
print("\nEnglish-centric, first merges:"); show(m_en, c_en)
print("\nLatin-weighted, first merges:"); show(m_lat, c_lat)

In [ ]:
vocab_en = {a + b for a, b in m_en}; vocab_lat = {a + b for a, b in m_lat}
only_lat = sorted(vocab_lat - vocab_en, key=len, reverse=True)[:25]
print("tokens the Latin-weighted vocabulary learned that the English-centric one never did:")
print("  ", " · ".join(t.decode(errors="replace") for t in only_lat))
def fert(merges, docs): return sum(len(encode(d[:1500], merges)) for d in docs) / sum(len(d[:1500].split()) for d in docs)
res = {name: {lang: fert(m, test[lang][:60]) for lang in test} for name, m in (("en-centric", m_en), ("latin-weighted", m_lat))}
for name, r in res.items(): print(f"{name:>15}: " + "  ".join(f"{l} {v:.2f}" for l, v in r.items()) + f"   es/en = {r['es']/r['en']:.2f}")
print("\nthe word 'entrenamiento' under each:", [t.decode(errors='replace') for t in encode(" entrenamiento", m_en)], [t.decode(errors='replace') for t in encode(" entrenamiento", m_lat)])

> **Exercise 3.1.** Raise `V` to 256 + 8,000 and re-run (a few minutes). Does the English-centric tokenizer's Spanish fertility catch up? It should narrow but not close — the merges are spent where the frequency is, and the Spanish share sets that.

## 4 · The production version, with pre-tokenization on purpose

The same two mixes through the `tokenizers` library at 32K: byte-level BPE, Llama-3-style digit groups, byte fallback. This is the tokenizer you would actually ship, and the fertility numbers to put in the dataset card.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
def train_prod(texts, vocab_size=32_000):
    tk = Tokenizer(models.BPE(byte_fallback=True))
    tk.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(pattern=r"\d{1,3}", behavior="isolated"),  # Llama-3 digit groups
        pre_tokenizers.ByteLevel(add_prefix_space=False)])
    tk.decoder = decoders.ByteLevel()
    tr = trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tk.train_from_iterator(texts, tr); return tk
t0 = time.time()
prod_en = train_prod(mix({"en": .95, "es": .03, "pt": .02}, n=1300)); prod_lat = train_prod(mix({"en": .60, "es": .25, "pt": .15}, n=1300))
print(f"trained in {time.time()-t0:.0f}s")
def fert_prod(tk, docs): return sum(len(tk.encode(d).ids) for d in docs) / sum(len(d.split()) for d in docs)
for name, tk in (("en-centric 32K", prod_en), ("latin-weighted 32K", prod_lat)):
    r = {l: fert_prod(tk, test[l]) for l in test}
    print(f"{name:>20}: " + "  ".join(f"{l} {v:.2f}" for l, v in r.items()) + f"   es/en = {r['es']/r['en']:.2f}  pt/en = {r['pt']/r['en']:.2f}")
print("\n'El entrenamiento de modelos cuesta 66000 horas.' →", prod_lat.encode("El entrenamiento de modelos cuesta 66000 horas.").tokens)
prod_lat.save("tokenizer_latin_32k.json")

## 5 · The architecture bill

Chapter 12.4: embedding parameters $2Vd$ (untied) and softmax FLOPs $2Vd$ per token, as a share of a model with $N_{nv}$ non-vocabulary parameters. Figure 12.4, recomputed for any model you like.

In [ ]:
def shares(N_nv, d, V, tied):
    emb = (1 if tied else 2) * V * d
    return emb / (N_nv + emb), (V * d) / (N_nv + V * d)        # param share, per-token FLOP share of the softmax
Vs = np.array([8e3, 16e3, 32e3, 64e3, 128e3, 256e3])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for N_nv, d, col, lab in ((1e9, 2048, AMBER, "1B, d=2048"), (7e9, 4096, NAVY, "8B, d=4096")):
    for tied, ls in ((False, "-"), (True, "--")):
        ps = [shares(N_nv, d, v, tied)[0] for v in Vs]; ax[0].plot(Vs, ps, color=col, ls=ls, label=f"{lab} {'tied' if tied else 'untied'}")
    ax[1].plot(Vs, [shares(N_nv, d, v, True)[1] for v in Vs], color=col, label=lab)
for a in ax: a.set_xscale("log", base=2); a.set_xlabel("vocabulary size")
ax[0].set_ylabel("share of parameters in embeddings"); ax[1].set_ylabel("share of per-token FLOPs in softmax"); ax[0].legend(fontsize=8); ax[1].legend()
plt.tight_layout()
for V_, name in ((128256, "Llama 3 8B"), (128256, "Llama 3.2 1B (tied)")):
    N_nv, d, tied = (7e9, 4096, False) if "8B" in name else (1e9, 2048, True)
    p_, f_ = shares(N_nv, d, V_, tied); print(f"{name}: embeddings {p_:.1%} of parameters, softmax {f_:.1%} of per-token FLOPs")

## 6 · Audit the tail of the vocabulary

Chapter 12.8: count how often each token of your 32K vocabulary appears in a held-out sample, by language. Rows that almost never appear are under-trained parameters; the per-language count is the fairness audit — did the merges go where the mix said?

In [ ]:
counts = collections.Counter(); by_lang = {l: collections.Counter() for l in test}
for l, docs in test.items():
    for d in docs:
        ids = prod_lat.encode(d).ids; counts.update(ids); by_lang[l].update(ids)
V_prod = prod_lat.get_vocab_size()
seen = np.array([counts.get(i, 0) for i in range(V_prod)])
print(f"tokens never seen in held-out: {np.mean(seen == 0):.1%};  seen < 5 times: {np.mean(seen < 5):.1%}  (of {V_prod})")
plt.figure(figsize=(6,3)); plt.hist(np.log10(seen + 1), bins=40, color=LIGHT, edgecolor=NAVY); plt.xlabel("log10(1 + held-out count)"); plt.ylabel("tokens"); plt.title("the vocabulary tail")
share = {l: len(by_lang[l]) / V_prod for l in by_lang}
print("fraction of the vocabulary used by each language's held-out text:", {l: f"{v:.1%}" for l, v in share.items()})

## 7 · Extend a base vocabulary and initialize the new rows

Chapter 12.5's cheapest route: append Spanish/Portuguese tokens from the Latin-weighted vocabulary to a base model (SmolLM2-135M), initialize each new row as the mean of the base tokens it replaces, and measure the fertility drop. (Continued pre-training is Chapter 20's job; here we only check that the rows start in a sensible place.)

In [ ]:
import torch
from transformers import AutoModelForCausalLM
BASE = "HuggingFaceTB/SmolLM2-135M"
btok = AutoTokenizer.from_pretrained(BASE); model = AutoModelForCausalLM.from_pretrained(BASE)
base_vocab = set(btok.get_vocab())
cands = [t for t, _ in sorted(prod_lat.get_vocab().items(), key=lambda kv: kv[1]) if t not in base_vocab and len(t) >= 4]
new_tokens = cands[:5000]
before = {l: measure(btok, test[l])[0] for l in test}
btok.add_tokens(new_tokens); old_n = model.get_input_embeddings().weight.shape[0]; model.resize_token_embeddings(len(btok))
E = model.get_input_embeddings().weight.data
with torch.no_grad():
    for i, t in enumerate(new_tokens):
        pieces = btok(t.replace("Ġ", " "), add_special_tokens=False).input_ids
        pieces = [p for p in pieces if p < old_n] or [btok.eos_token_id]
        E[old_n + i] = E[pieces].mean(0)
after = {l: measure(btok, test[l])[0] for l in test}
for l in test: print(f"{l}: fertility {before[l]:.2f} → {after[l]:.2f}  ({(after[l]/before[l]-1):+.0%})")
print("new rows initialized:", len(new_tokens), "| embedding matrix:", tuple(E.shape))

> **Exercise 7.1.** Compare three initializations for the new rows — random normal, the mean of the fragments (above), and the embedding of the *first* fragment — by the base model's loss on 50 Spanish documents before any training. The mean should be lowest; that gap is what continued pre-training would otherwise have to close.
>
> **Exercise 7.2.** Write the tax table (§1) and the token-share mix (§2) into Lab 10's `README.md` under a "Tokenization" heading. From here on, the book counts in tokens.

## What you reproduced

* fertility, bytes/token and the tax ratio for four open tokenizers on your own en/es/pt slices (Figure 12.1, measured);
* the data mix re-stated in tokens;
* the chapter's BPE, trained from scratch on two mixes, with the real merge tables and the Spanish tokens only one of them learns;
* a production 32K BPE with Llama-3 digit groups, saved to `tokenizer_latin_32k.json`;
* the embedding and softmax bill versus vocabulary size (Figure 12.4);
* the vocabulary-tail and per-language usage audit;
* vocabulary extension with fragment-mean initialization and its fertility drop.

This closes Phase 2's labs: Lab 10 built the slice, Lab 11 validated it, Lab 12 counted it in tokens, Lab 27 grew it.